# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [2]:
# Write your code below.
%load_ext dotenv
%dotenv 


In [2]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [3]:
import os
from glob import glob

# Write your code below.

os.environ["PRICE_DATA"] = r"C:\Users\ekadw\Documents\GitHub\production\05_src\data\prices_csv\etfs"

price_data = os.getenv("PRICE_DATA")
print("PRICE_DATA is:", price_data)

stock_files = glob(os.path.join(price_data, "*.csv"))
print(stock_files)

PRICE_DATA is: C:\Users\ekadw\Documents\GitHub\production\05_src\data\prices_csv\etfs
['C:\\Users\\ekadw\\Documents\\GitHub\\production\\05_src\\data\\prices_csv\\etfs\\AAAU.csv', 'C:\\Users\\ekadw\\Documents\\GitHub\\production\\05_src\\data\\prices_csv\\etfs\\AADR.csv', 'C:\\Users\\ekadw\\Documents\\GitHub\\production\\05_src\\data\\prices_csv\\etfs\\AAXJ.csv', 'C:\\Users\\ekadw\\Documents\\GitHub\\production\\05_src\\data\\prices_csv\\etfs\\ABEQ.csv', 'C:\\Users\\ekadw\\Documents\\GitHub\\production\\05_src\\data\\prices_csv\\etfs\\ACES.csv', 'C:\\Users\\ekadw\\Documents\\GitHub\\production\\05_src\\data\\prices_csv\\etfs\\ACIO.csv', 'C:\\Users\\ekadw\\Documents\\GitHub\\production\\05_src\\data\\prices_csv\\etfs\\ACSG.csv', 'C:\\Users\\ekadw\\Documents\\GitHub\\production\\05_src\\data\\prices_csv\\etfs\\ACSI.csv', 'C:\\Users\\ekadw\\Documents\\GitHub\\production\\05_src\\data\\prices_csv\\etfs\\ACT.csv', 'C:\\Users\\ekadw\\Documents\\GitHub\\production\\05_src\\data\\prices_csv\\e

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [10]:
import os
import dask.dataframe as dd

# 1. Get your PRICE_DATA path
price_data = os.getenv("PRICE_DATA")
print("PRICE_DATA is:", price_data)

# 2. Load CSVs with path info (so we can get ticker from filename)
dd_px = dd.read_csv(
    os.path.join(price_data, "*.csv"),
    parse_dates=["Date"],
    assume_missing=True,
    include_path_column="file_path"
)

# 3. Extract ticker from filename
dd_px["ticker"] = dd_px["file_path"].map_partitions(
    lambda s: s.apply(lambda x: os.path.basename(x).replace(".csv", ""))
)
dd_px = dd_px.drop("file_path", axis=1)

print(dd_px.head())   # sanity check


PRICE_DATA is: C:\Users\ekadw\Documents\GitHub\production\05_src\data\prices_csv\etfs
        Date   Open   High    Low  Close  Adj Close    Volume ticker
0 2018-08-15  11.84  11.84  11.74  11.74      11.74   27300.0   AAAU
1 2018-08-16  11.78  11.80  11.74  11.74      11.74  428400.0   AAAU
2 2018-08-17  11.80  11.82  11.77  11.82      11.82   52400.0   AAAU
3 2018-08-20  11.88  11.91  11.85  11.90      11.90   28700.0   AAAU
4 2018-08-21  11.92  11.95  11.89  11.93      11.93   30600.0   AAAU


In [13]:
def add_features(df):
    return df.assign(
        Close_lag_1 = df['Close'].shift(1),
        Adj_Close_lag_1 = df['Adj_Close'].shift(1),
        returns = (df['Close'] / df['Close'].shift(1)) - 1,
        hi_lo_range = df['High'] - df['Low']
    )

# define schema (meta)
meta = dd.utils.make_meta({
    'Date': 'datetime64[ns]',
    'Open': 'f8',
    'High': 'f8',
    'Low': 'f8',
    'Close': 'f8',
    'Adj_Close': 'f8',
    'Volume': 'f8',
    'ticker': 'str',
    'Close_lag_1': 'f8',
    'Adj_Close_lag_1': 'f8',
    'returns': 'f8',
    'hi_lo_range': 'f8'
})

dd_px = dd_px.rename(columns=lambda x: x.replace(" ", "_"))

dd_feat = dd_px.groupby("ticker", group_keys=False, observed=True).apply(add_features, meta=meta)

print(dd_feat.head())


        Date       Open       High        Low      Close  Adj_Close   Volume  \
0 2016-09-15  24.955000  25.200001  24.860001  25.180000  24.101143   6200.0   
0 2019-10-01  25.030001  25.030001  24.580000  24.580000  24.447256  13900.0   
0 2017-09-25  25.230000  25.230000  25.156000  25.156000  23.331795    200.0   
0 2009-04-16  60.610001  61.040001  60.610001  61.000000  54.692841   1500.0   
0 2018-02-15  80.000000  80.000000  76.550003  76.879997  73.497726    300.0   

  ticker  Close_lag_1  Adj_Close_lag_1  returns  hi_lo_range  
0   FDMO          NaN              NaN      NaN     0.340000  
0    OVS          NaN              NaN      NaN     0.450001  
0   PBSM          NaN              NaN      NaN     0.073999  
0    TYO          NaN              NaN      NaN     0.430000  
0   VFVA          NaN              NaN      NaN     3.449997  


In [14]:
print(dd_px.columns)


Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj_Close', 'Volume',
       'ticker'],
      dtype='object')


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [15]:
# Write your code below.

# 1. Compute the full Dask DataFrame into pandas
pdf = dd_feat.compute()
print(type(pdf))   # should show <class 'pandas.core.frame.DataFrame'>

# 2. Sort by ticker and Date (important for rolling)
pdf = pdf.sort_values(["ticker", "Date"])

# 3. Add 10-day moving average of returns (per ticker)
pdf["ma10_returns"] = pdf.groupby("ticker")["returns"].transform(lambda x: x.rolling(10).mean())

print(pdf.head(15))


<class 'pandas.core.frame.DataFrame'>


C:\Users\ekadw\AppData\Local\Temp\ipykernel_12296\3972239113.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pdf["ma10_returns"] = pdf.groupby("ticker")["returns"].transform(lambda x: x.rolling(10).mean())


         Date   Open   High    Low  Close  Adj_Close     Volume ticker  \
0  2018-08-15  11.84  11.84  11.74  11.74      11.74    27300.0   AAAU   
1  2018-08-16  11.78  11.80  11.74  11.74      11.74   428400.0   AAAU   
2  2018-08-17  11.80  11.82  11.77  11.82      11.82    52400.0   AAAU   
3  2018-08-20  11.88  11.91  11.85  11.90      11.90    28700.0   AAAU   
4  2018-08-21  11.92  11.95  11.89  11.93      11.93    30600.0   AAAU   
5  2018-08-22  11.98  11.98  11.94  11.96      11.96   101200.0   AAAU   
6  2018-08-23  11.91  11.92  11.85  11.85      11.85    54800.0   AAAU   
7  2018-08-24  11.97  12.08  11.96  12.05      12.05   106500.0   AAAU   
8  2018-08-27  12.06  12.13  12.06  12.10      12.10   295100.0   AAAU   
9  2018-08-28  12.13  12.13  12.01  12.01      12.01    30400.0   AAAU   
10 2018-08-29  12.06  12.06  12.03  12.05      12.05    31800.0   AAAU   
11 2018-08-30  12.03  12.03  11.97  12.01      12.01   804500.0   AAAU   
12 2018-08-31  12.04  12.05  11.99  12

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

Answer: It depends on the data size. If the data size is small, it is better to convert to pandas since it is simpler. But if the data is large and can not put directly into memory, we can use Dask. I heard that using Dask is trickier than using Pandas.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.